# RAG Pipeline Main Execution

This notebook runs the full RAG pipeline (Baseline and Fusion) on a golden dataset and saves the responses for evaluation.

In [ ]:
import torch
import datetime
import gc
import json
import os
import pickle
from qdrant_client import QdrantClient
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from tqdm.auto import tqdm
from dotenv import load_dotenv

# Load environment variables
load_dotenv("../.env")

In [ ]:
# Configuration Dictionary
config = {
    "MODEL_ID": "Qwen/Qwen2.5-7B-Instruct", # Example, adjust as needed
    "EMBEDDING_MODEL": "google/embeddinggemma-300m",
    "QDRANT_URL": os.getenv("QDRANT_URL"),
    "QDRANT_API_KEY": os.getenv("QDRANT_API_KEY"),
    "OPENAI_API_KEY": os.getenv("OPENAI_API_KEY"),
    "HF_TOKEN": os.getenv("HF_TOKEN"),
    "LANGSMITH_ENDPOINT": os.getenv("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com"),
    "LANGSMITH_PROJECT": os.getenv("LANGSMITH_PROJECT", "medical-rag"),
    "LANGSMITH_API_KEY": os.getenv("LANGSMITH_API_KEY"),
    "COLLECTION_NAME": "medical_reports_v3",
    "TOP_K": 3,
    "GOLDEN_DATASET_PATH": "../knowledge-base/nutrition_qa_golden_dataset.json",
    "PICKLE_PATH": "../knowledge-base/2026_04_29__1545_medical_documents_enriched.pkl"
}

In [ ]:
class RAGPipelineBase:
    """Base class to avoid code duplication between Baseline and Fusion."""
    
    GEN_PROMPT_TEMPLATE = """<|im_start|>system
Anda adalah asisten medis profesional. Jawablah pertanyaan klinis berdasarkan KONTEKS yang diberikan.

ATURAN:
1. Jika konteks berasal dari dokumen yang berbeda (sumber berbeda), jawablah dengan hati-hati dan jangan mencampuradukkan data antar pasien.
2. Jawablah secara detail namun tetap pada intinya, pastikan semua poin teknis medis yang relevan dari konteks disebutkan.
3. Jika informasi pendukung tidak lengkap di konteks, jawablah sesuai informasi yang ada tanpa mengarang fakta medis tambahan. Jika benar-benar tidak ada informasi, nyatakan "Informasi tidak lengkap."
<|im_end|>
<|im_start|>user
KONTEKS:
{context}

PERTANYAAN:
{question}
<|im_end|>
<|im_start|>assistant
"""

    def __init__(self, config, llm=None, vectorstore=None):
        self.config = config
        self.setup_environment()
        
        if llm:
            self.llm = llm
        else:
            self.setup_llm()
            
        if vectorstore:
            self.vectorstore = vectorstore
        else:
            self.setup_vectorstore()
        
        self.setup_bm25()

    def setup_environment(self):
        os.environ["LANGCHAIN_TRACING_V2"] = "true"
        os.environ["LANGCHAIN_ENDPOINT"] = self.config.get('LANGSMITH_ENDPOINT', '')
        os.environ["LANGCHAIN_PROJECT"] = self.config.get('LANGSMITH_PROJECT', '')
        os.environ["LANGCHAIN_API_KEY"] = self.config.get('LANGSMITH_API_KEY', '')
        os.environ["OPENAI_API_KEY"] = self.config.get('OPENAI_API_KEY', '')
        os.environ["HF_TOKEN"] = self.config.get('HF_TOKEN', '')

    def setup_llm(self):
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )
        tokenizer = AutoTokenizer.from_pretrained(self.config['MODEL_ID'])
        model = AutoModelForCausalLM.from_pretrained(
            self.config['MODEL_ID'],
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
        
        text_pipe = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            max_new_tokens=512,
            temperature=0.1,
            repetition_penalty=1.1,
            return_full_text=False,
            stop_sequence=["<|im_end|>", "<|endoftext|>"]
        )
        self.llm = HuggingFacePipeline(pipeline=text_pipe)

    def setup_vectorstore(self):
        self.client = QdrantClient(url=self.config['QDRANT_URL'], api_key=self.config['QDRANT_API_KEY'], timeout=300)
        
        embedding_bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )

        self.embedding_model = HuggingFaceEmbeddings(
            model_name=self.config['EMBEDDING_MODEL'],
            model_kwargs={
                'device': 'cuda', 
                'token': self.config['HF_TOKEN'],
                'trust_remote_code': True,
                'quantization_config': embedding_bnb_config,
                'torch_dtype': torch.float16,
            }
        )
        self.vectorstore = QdrantVectorStore(
            client=self.client,
            collection_name=self.config['COLLECTION_NAME'],
            embedding=self.embedding_model,
            content_payload_key="page_content",
        )

    def setup_bm25(self):
        pickle_path = self.config.get('PICKLE_PATH')
        if os.path.exists(pickle_path):
            with open(pickle_path, 'rb') as f:
                docs = pickle.load(f)
            self.bm25_retriever = BM25Retriever.from_documents(docs)
            self.bm25_retriever.k = self.config.get('TOP_K', 3)
        else:
            print(f"Warning: Pickle file not found at {pickle_path}.")
            self.bm25_retriever = None

    @staticmethod
    def format_docs(docs):
        formatted = []
        for i, doc in enumerate(docs):
            source = doc.metadata.get('source', 'Unknown')
            formatted.append(f"--- DOKUMEN {i+1} (Sumber: {source}) ---\n{doc.page_content}")
        return "\n\n".join(formatted)

    def hybrid_search(self, query, k=3, filter_dict=None):
        instruction = "Representasikan pertanyaan medis ini untuk pencarian rekam medis yang relevan: "
        if filter_dict:
            from qdrant_client.http import models as rest
            must_conditions = [
                rest.FieldCondition(key=f"metadata.{k}", match=rest.MatchValue(value=v))
                for k, v in filter_dict.items() if v
            ]
            qdrant_filter = rest.Filter(must=must_conditions) if must_conditions else None
            vector_docs = self.vectorstore.similarity_search(instruction + query, k=k, filter=qdrant_filter)
        else:
            vector_docs = self.vectorstore.similarity_search(instruction + query, k=k)

        if self.bm25_retriever:
            bm25_docs = self.bm25_retriever.invoke(query)
            if filter_dict:
                bm25_docs = [
                    doc for doc in bm25_docs 
                    if all(doc.metadata.get(mk) == mv for mk, mv in filter_dict.items())
                ]
            bm25_docs = bm25_docs[:k]
        else:
            bm25_docs = []

        combined = {doc.page_content: doc for doc in vector_docs + bm25_docs}
        return list(combined.values())[:k]

class RAGBaseline(RAGPipelineBase):
    def __init__(self, config, llm=None, vectorstore=None):
        super().__init__(config, llm, vectorstore)
        self.setup_generation_chain()

    def setup_generation_chain(self):
        self.prompt = ChatPromptTemplate.from_template(self.GEN_PROMPT_TEMPLATE)
        self.generation_chain = self.prompt | self.llm | StrOutputParser()

    def run(self, query, filter_dict=None):
        docs = self.hybrid_search(query, k=self.config.get('TOP_K', 3), filter_dict=filter_dict)
        formatted_context = self.format_docs(docs)
        answer = self.generation_chain.invoke({"context": formatted_context, "question": query})
        
        return {
            "answer": answer,
            "contexts": docs,
            "ids": [doc.metadata.get('chunk_id') for doc in docs]
        }

class RAGFusion(RAGPipelineBase):
    def __init__(self, config, llm=None, vectorstore=None):
        super().__init__(config, llm, vectorstore)
        self.top_n = self.config.get('TOP_K', 3)
        self.setup_fusion_pipeline()
        self.setup_generation_chain()

    def reciprocal_rank_fusion(self, results, k=60):
        fused_scores = {}
        for docs in results:
            for rank, doc in enumerate(docs):
                doc_str = json.dumps({"page_content": doc.page_content, "metadata": doc.metadata})
                if doc_str not in fused_scores:
                    fused_scores[doc_str] = 0
                fused_scores[doc_str] += 1 / (rank + k)
        
        reranked = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
        final_docs = []
        for doc_json, _ in reranked[:self.top_n]:
            data = json.loads(doc_json)
            final_docs.append(Document(page_content=data['page_content'], metadata=data['metadata']))
        return final_docs

    def setup_fusion_pipeline(self):
        query_gen_template = """<|im_start|>system
Anda adalah ahli pencarian rekam medis. Tugas Anda:
1. Bedah pertanyaan pengguna menjadi sub-pertanyaan yang lebih sederhana jika kompleks.
2. Buat variasi istilah medis (misal: "BBLR" menjadi "Berat Badan Lahir Rendah").
3. JANGAN hilangkan informasi penting pengidentifikasi pasien (diagnosis, usia, gender).
Berikan 3 variasi pertanyaan, satu per baris, tanpa angka.
<|im_end|>
<|im_start|>user
{question}
<|im_end|>
<|im_start|>assistant
"""
        prompt = ChatPromptTemplate.from_template(query_gen_template)
        self.query_generator = (
            prompt 
            | ChatOpenAI(model="gpt-4o-mini", temperature=0.2) 
            | StrOutputParser() 
            | (lambda x: [q.strip() for q in x.split("\n") if q.strip()])
        )

    def setup_generation_chain(self):
        self.prompt = ChatPromptTemplate.from_template(self.GEN_PROMPT_TEMPLATE)
        self.generation_chain = self.prompt | self.llm | StrOutputParser()

    def run(self, query, filter_dict=None):
        generated_queries = self.query_generator.invoke({"question": query})
        all_queries = [query] + generated_queries
        
        all_results = []
        for q in all_queries:
            docs = self.hybrid_search(q, k=self.top_n * 2, filter_dict=filter_dict)
            all_results.append(docs)
                
        fused_docs = self.reciprocal_rank_fusion(all_results)
        formatted_context = self.format_docs(fused_docs)
        answer = self.generation_chain.invoke({
            "context": formatted_context,
            "question": query
        })
        
        return {
            "answer": answer,
            "contexts": fused_docs,
            "ids": [doc.metadata.get('chunk_id') for doc in fused_docs]
        }

In [ ]:
print("Menginisialisasi Model dan Vector Database...")
base_pipeline = RAGPipelineBase(config=config)

rag_baseline = RAGBaseline(config=config, llm=base_pipeline.llm, vectorstore=base_pipeline.vectorstore)
rag_fusion = RAGFusion(config=config, llm=base_pipeline.llm, vectorstore=base_pipeline.vectorstore)
print("Inisialisasi selesai!\n")

In [ ]:
current_date = datetime.datetime.now().strftime("%Y%m%d_%H%M")
input_file = config["GOLDEN_DATASET_PATH"]
output_file = f"../{current_date}_rag_responses_dataset_k{config['TOP_K']}.jsonl"

with open(input_file, 'r', encoding='utf-8') as f:
    dataset = json.load(f)

print(f"Processing {len(dataset)} questions...")

with open(output_file, 'w', encoding='utf-8') as f_out:
    for item in tqdm(dataset):
        original_question = item['question']
        
        diagnosis = item['chunk_metadata'][0].get('medical_diagnosis', '')
        query_to_run = f"Diagnosis: {diagnosis}. Pertanyaan: {original_question}"
        filter_dict = {"medical_diagnosis": diagnosis} if diagnosis else None

        # Baseline execution
        baseline_data = {"answer": "ERROR", "ids": [], "contexts": [], "metadata": []}
        try:
            res_b = rag_baseline.run(query_to_run, filter_dict=filter_dict)
            baseline_data = {
                "answer": res_b['answer'],
                "ids": res_b['ids'],
                "contexts": [d.page_content for d in res_b['contexts']],
                "metadata": [d.metadata for d in res_b['contexts']]
            }
        except Exception as e:
            print(f"Baseline error: {e}")

        # Fusion execution
        fusion_data = {"answer": "ERROR", "ids": [], "contexts": [], "metadata": []}
        try:
            res_f = rag_fusion.run(query_to_run, filter_dict=filter_dict)
            fusion_data = {
                "answer": res_f['answer'],
                "ids": res_f['ids'],
                "contexts": [d.page_content for d in res_f['contexts']],
                "metadata": [d.metadata for d in res_f['contexts']]
            }
        except Exception as e:
            print(f"Fusion error: {e}")

        result = {
            "question": original_question,
            "ground_truth": item['ground_truth'],
            "chunk_metadata": item['chunk_metadata'],
            "baseline": baseline_data,
            "fusion": fusion_data
        }

        f_out.write(json.dumps(result, ensure_ascii=False) + '\n')
        f_out.flush()

        gc.collect()
        torch.cuda.empty_cache()